<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_2_convolucion.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_2_convolucion.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F03_vision%2Fnotebooks%2Fsesion_03_2_convolucion.ipynb)

# SI7011 · Sesión 03 · 2. La convolución por dentro
**Juan David Martínez Vargas · Universidad EAFIT**

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | convolución escrita a mano frente a `F.conv2d` | que dan exactamente lo mismo |
| 2 | kernels hechos a mano sobre una foto | qué patrón enciende cada uno |
| 3 | kernel, padding y stride | la fórmula del tamaño de salida |
| 4 | parámetros y cómputo | pocos pesos, muchas operaciones |
| 5 | desplazar la imagen | equivarianza, y cuándo se rompe |
| 6 | el gradiente de una salida | el campo receptivo, medido |
| 7 | la CNN del notebook 1, entrenada | los kernels que aprende |

Corre en CPU en 2–3 minutos (Colab, Kaggle, Lightning o local).

In [ ]:
import time
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110})
print('PyTorch', torch.__version__, '· device', device)

## Datos: CIFAR-10, como en el notebook 1
Guardamos también las imágenes **sin normalizar** (`R_train`) para mostrarlas.

In [ ]:
train_set = datasets.CIFAR10('data', train=True, download=True)
CLASSES = train_set.classes
to_tensor = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float() / 255
X = to_tensor(train_set.data); y = torch.tensor(train_set.targets)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
R_train, y_train = X[perm[:10_000]], y[perm[:10_000]]
R_val, y_val = X[perm[45_000:]], y[perm[45_000:]]

mean, std = R_train.mean(dim=(0, 2, 3)), R_train.std(dim=(0, 2, 3))
norm = lambda X: (X - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
X_train, X_val = norm(R_train), norm(R_val)
X_train.shape

## 1. Convolución a mano
La definición de la diapositiva *Convolution as a sliding dot product*:

$$y_{i,j} = \sum_{u=0}^{K-1}\sum_{v=0}^{K-1} k_{u,v}\, x_{i+u,\;j+v}$$

Usamos la misma imagen de 6×6 y el mismo kernel de la animación: una franja brillante vertical y un detector de bordes verticales.

In [ ]:
x6 = torch.tensor([[1, 1, 7, 7, 1, 1],
                   [1, 2, 8, 7, 2, 1],
                   [0, 1, 7, 8, 1, 0],
                   [1, 1, 8, 7, 1, 1],
                   [2, 1, 7, 7, 1, 2],
                   [1, 1, 8, 8, 1, 1]], dtype=torch.float32)
k3 = torch.tensor([[-1, 0, 1]] * 3, dtype=torch.float32)

def conv2d_a_mano(x, k):
    K = k.shape[0]
    H_out, W_out = x.shape[0] - K + 1, x.shape[1] - K + 1
    y = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            y[i, j] = (k * x[i:i + K, j:j + K]).sum()      # producto punto con la ventana
    return y

y_mano = conv2d_a_mano(x6, k3)
y_mano

In [ ]:
y_torch = F.conv2d(x6.view(1, 1, 6, 6), k3.view(1, 1, 3, 3)).squeeze()
print('¿iguales?', torch.equal(y_mano, y_torch))

`F.conv2d` espera tensores $N \times C \times H \times W$ y pesos $C_\text{out} \times C_\text{in} \times K \times K$; de ahí los `view`. No voltea el kernel: calcula una correlación cruzada, como la fórmula. Los valores positivos aparecen en el borde izquierdo de la franja y los negativos en el derecho.

## 2. Kernels hechos a mano sobre una foto
Los mismos cuatro kernels de la figura de la diapositiva, aplicados a una imagen de CIFAR-10 en escala de grises. `padding=1` mantiene el tamaño de 32×32.

In [ ]:
KERNELS = {
    'bordes verticales': [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    'bordes horizontales': [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    'suavizado': [[1 / 9] * 3] * 3,
    'realce': [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}
gray = R_train[7].mean(0, keepdim=True).unsqueeze(0)          # 1 × 1 × 32 × 32
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
axes[0].imshow(gray[0, 0], cmap='gray'); axes[0].set_title(CLASSES[y_train[7]])
for ax, (name, k) in zip(axes[1:], KERNELS.items()):
    out = F.conv2d(gray, torch.tensor(k, dtype=torch.float32).view(1, 1, 3, 3), padding=1)[0, 0]
    ax.imshow(out, cmap='RdBu_r' if 'bordes' in name else 'gray'); ax.set_title(name, fontsize=9)
for ax in axes: ax.axis('off')
plt.show()

## 3. Kernel, padding y stride
$$H_\text{out} = \left\lfloor \frac{H + 2p - K}{s} \right\rfloor + 1$$
Comparamos la fórmula con lo que devuelve `nn.Conv2d` para una imagen de 3×32×32.

In [ ]:
out_size = lambda H, K, p=0, s=1: (H + 2 * p - K) // s + 1
x1 = X_train[:1]
print(' K  p  s   forma real      fórmula')
for K, p, s in [(3, 0, 1), (3, 1, 1), (3, 1, 2), (5, 2, 1), (1, 0, 1), (16, 0, 16)]:
    shape = tuple(nn.Conv2d(3, 8, K, stride=s, padding=p)(x1).shape[1:])
    print(f'{K:2d} {p:2d} {s:2d}   {str(shape):14s}  {out_size(32, K, p, s)}')

La última fila, $K = s = 16$, parte la imagen en ventanas que no se solapan: es el *patch embedding* del notebook 5.

**Respuesta a la diapositiva *Prediction before experiment*:**

In [ ]:
conv = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
a = torch.zeros(1, 64, 32, 32)
out = conv(a)
print('forma de salida', tuple(out.shape[1:]))
print('parámetros', sum(p.numel() for p in conv.parameters()))
print(f'activaciones: {a.numel():,} → {out.numel():,}')

## 4. Parámetros y cómputo
Para una entrada y una salida de 32×32 con un solo canal (diapositiva *Dense, local, shared*, animación de la diapositiva siguiente):

In [ ]:
P = lambda m: sum(p.numel() for p in m.parameters())
dense = nn.Linear(32 * 32, 32 * 32)
local = 32 * 32 * 9 + 32 * 32               # 9 pesos propios + 1 sesgo por cada salida
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
print(f'densa: {P(dense):,} · localmente conectada: {local:,} · convolución: {P(conv)}')

Los parámetros de una convolución no dependen del tamaño de la imagen, pero el **cómputo** sí: los mismos pesos se aplican en cada posición. Contamos las operaciones de la CNN del notebook 1 con un *hook* que mira la salida de cada `Conv2d`.

In [ ]:
def small_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(64 * 4 * 4, 10),
    )

cnn, flops = small_cnn(), []
def count(m, inp, out):
    flops.append((m, 2 * m.out_channels * m.in_channels * m.kernel_size[0] ** 2 * out.shape[2] * out.shape[3]))
hooks = [m.register_forward_hook(count) for m in cnn if isinstance(m, nn.Conv2d)]
cnn(X_train[:1]); [h.remove() for h in hooks]
for m, f in flops:
    print(f'{str(m):60s} {P(m):7,} parámetros · {f / 1e6:5.1f} MFLOPs')

La primera convolución tiene apenas 896 pesos, pero como trabaja sobre 32×32 posiciones cuesta casi 2 MFLOPs. La tercera tiene 41 veces más pesos y cuesta menos de 3 veces más, porque trabaja sobre 8×8. Por eso las CNN reducen la resolución a medida que agregan canales.

## 5. Equivarianza: desplazar la imagen
Si se desplaza la entrada, el mapa de salida se desplaza lo mismo:
$\operatorname{conv}(\operatorname{shift}(x)) = \operatorname{shift}(\operatorname{conv}(x))$. Lo comprobamos lejos de los bordes, porque `torch.roll` hace que lo que sale por un lado entre por el otro.

In [ ]:
torch.manual_seed(0)
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
shift = lambda t, d: torch.roll(t, shifts=(d, d), dims=(2, 3))
inner = (slice(None), slice(None), slice(6, -6), slice(6, -6))
x1 = X_train[:1]
with torch.no_grad():
    err = (conv(shift(x1, 2)) - shift(conv(x1), 2))[inner].abs().max()
print(f'stride 1, desplazar 2 píxeles: error máximo {err:.1e}')

**Cambia un argumento:** `stride=2`. Ahora la salida tiene la mitad de posiciones. Un desplazamiento de 2 píxeles en la entrada equivale a 1 en la salida, pero un desplazamiento de 1 píxel no corresponde a ningún desplazamiento entero de la salida.

In [ ]:
torch.manual_seed(0)
conv2 = nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1)
inner2 = (slice(None), slice(None), slice(3, -3), slice(3, -3))
with torch.no_grad():
    for d_in in [2, 1]:
        a = conv2(shift(x1, d_in))
        errs = {d_out: (a - shift(conv2(x1), d_out))[inner2].abs().max().item() for d_out in [0, 1]}
        print(f'stride 2, desplazar {d_in} → error con desplazamiento de salida 0: {errs[0]:.2f} · 1: {errs[1]:.2e}')

Con desplazamiento 2 el error es cero si la salida se desplaza 1. Con desplazamiento 1 ningún desplazamiento de la salida lo explica: el stride rompe la equivarianza en parte.

## 6. El campo receptivo, medido
El campo receptivo de una unidad es la región de la entrada de la que depende. Una forma directa de verlo es **el gradiente**: los píxeles de entrada con gradiente distinto de cero son los que afectan a esa unidad.

Usamos convoluciones lineales con todos los pesos en 1. El campo receptivo depende solo de la arquitectura, no de los valores de los pesos.

In [ ]:
def ones_conv(stride):
    c = nn.Conv2d(1, 1, kernel_size=3, stride=stride, bias=False)
    nn.init.ones_(c.weight)
    return c

def receptive_field(depth, stride, size=33):
    net = nn.Sequential(*[ones_conv(stride) for _ in range(depth)])
    x = torch.zeros(1, 1, size, size, requires_grad=True)
    out = net(x)
    c = out.shape[-1] // 2
    out[0, 0, c, c].backward()                    # gradiente de UNA unidad de salida
    rows = (x.grad[0, 0].abs() > 0).any(1).nonzero()
    return int(rows.max() - rows.min() + 1), x.grad[0, 0]

for s in [1, 2]:
    print(f'stride {s}: r =', [receptive_field(d, s)[0] for d in [1, 2, 3]])

Son los valores de la fórmula $r_l = r_{l-1} + (K_l - 1)\prod_{i<l} s_i$ y de la animación: 3, 5 y 7 con stride 1; 3, 7 y 15 con stride 2.

In [ ]:
_, g = receptive_field(3, 1)
plt.figure(figsize=(3.2, 3.2)); plt.imshow(g[11:22, 11:22], cmap='Greens')
plt.title('gradiente de una unidad\n(3 capas 3×3, stride 1)', fontsize=9); plt.axis('off'); plt.show()

Dentro del campo receptivo de 7×7 el gradiente no es uniforme: el centro llega a la salida por muchos más caminos que los bordes. Por eso el campo receptivo *efectivo* es menor que el teórico.

## 7. Los kernels que aprende una CNN
Entrenamos la CNN del notebook 1 durante 5 épocas con el mismo loop y miramos los 32 kernels de la primera capa. Cada uno tiene forma $3 \times 3 \times 3$, así que se puede mostrar como una pequeña imagen en color.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
torch.manual_seed(0)
cnn = small_cnn().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(5):
    cnn.train()
    order = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 128):
        b = order[i:i + 128]
        xb, yb = X_train[b].to(device), y_train[b].to(device)
        loss = loss_fn(cnn(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
cnn.eval()
with torch.no_grad():
    acc = torch.cat([cnn(X_val[i:i + 1000].to(device)).argmax(1).cpu() for i in range(0, len(X_val), 1000)]).eq(y_val).float().mean()
print(f'val_acc {acc:.3f} · {time.time() - t0:.0f} s')

In [ ]:
W = cnn[0].weight.detach().cpu()                  # 32 × 3 × 3 × 3
W = (W - W.amin(dim=(1, 2, 3), keepdim=True)) / (W.amax(dim=(1, 2, 3), keepdim=True) - W.amin(dim=(1, 2, 3), keepdim=True))
fig, axes = plt.subplots(2, 16, figsize=(11, 1.6))
for ax, w in zip(axes.flat, W):
    ax.imshow(w.permute(1, 2, 0)); ax.axis('off')
plt.suptitle('kernels de la primera capa (cada uno 3×3×3)', fontsize=9); plt.show()

In [ ]:
with torch.no_grad():
    maps = F.relu(cnn[0](X_train[7:8].to(device)))[0].cpu()
fig, axes = plt.subplots(1, 9, figsize=(11, 1.6))
axes[0].imshow(R_train[7].permute(1, 2, 0)); axes[0].set_title('entrada', fontsize=8)
for k, ax in enumerate(axes[1:]):
    ax.imshow(maps[k], cmap='viridis'); ax.set_title(f'canal {k}', fontsize=8)
for ax in axes: ax.axis('off')
plt.show()

Muchos kernels responden a bordes con una orientación o a colores particulares, los mismos patrones que pusimos a mano en la sección 2, pero ahora **aprendidos**. Con 3×3 los kernels son pequeños; en el notebook 4 veremos los de 7×7 de una ResNet entrenada en ImageNet.

## Tu turno · 10 minutos
1. En la sección 1, cambia el kernel por su transpuesta (`k3.T`). ¿Qué patrón detecta ahora en `x6`? ¿Por qué la salida es casi cero?
2. En la sección 3, ¿qué combinación de `K`, `p` y `s` reduce 32×32 a 8×8 en una sola capa? Compruébalo con la fórmula y con `nn.Conv2d`.
3. En la sección 6, mide el campo receptivo de dos capas: una con kernel 5 y otra con kernel 3. ¿Coincide con la fórmula?
4. En la sección 6, agrega un `nn.MaxPool2d(2)` entre las capas. ¿Cuánto crece el campo receptivo? ¿Se comporta como una capa con stride 2?

**Para pensar.** La sección 4 dice que una convolución tiene pocos parámetros y mucho cómputo. ¿Qué le conviene más a una red que va a correr en un teléfono, y por qué la respuesta no es solo "menos parámetros"?

In [ ]:
# Experimento 1

### Referencias
- Dumoulin y Visin (2016), *A guide to convolution arithmetic for deep learning*, arXiv:1603.07285.
- Luo et al. (2016), *Understanding the Effective Receptive Field in Deep Convolutional Neural Networks*, NeurIPS.
- Zhang et al., *Dive into Deep Learning*, cap. 7 (convoluciones).